## c16 - Timber Stock Dataset Generation
Generates new and reclaimed stock CSVs for donor buildings A and B, plus a small development subset. Outputs: `complete_timber_A.csv`, `complete_timber_B.csv`, `complete_timber_new.csv`.

## 1. Stock generation
Generates new stock once (shared baseline) and reclaimed stock for donor A and B separately, then combines and saves.

In [ ]:
import pandas as pd
import config
from c16_generation_timber import generate_new_stock, generate_reclaimed_stock

efficient = True  # True = minimum transport distance; False = realistic randomised distances

# ── Generate New Stock (once — shared across all output files) ────────────────
df_new_stock = generate_new_stock(efficient)
display(df_new_stock.head(3))

# ── Generate Reclaimed Stock — Donor A and B ──────────────────────────────────
# "A" — 3-storey Dutch residential (hardcoded spans: 1800–4500 mm)
# "B" — commercial/industrial (spans derived from structure's min/max length statistics)
df_reclaimed_A = generate_reclaimed_stock(donor_building="A")
df_reclaimed_B = generate_reclaimed_stock(donor_building="B")

for label, df_rs in [("A", df_reclaimed_A), ("B", df_reclaimed_B)]:
    print(f"\nDonor {label}: {len(df_rs)} elements  |  "
          f"length {df_rs['Length'].min():.0f}–{df_rs['Length'].max():.0f} mm  |  "
          f"mean {df_rs['Length'].mean():.0f} mm")

# ── Combine: same new stock base + each donor's reclaimed pool ────────────────
df_combined_A = pd.concat([df_new_stock, df_reclaimed_A], ignore_index=True)
df_combined_B = pd.concat([df_new_stock, df_reclaimed_B], ignore_index=True)

# ── Save ──────────────────────────────────────────────────────────────────────
files = {
    "complete_timber_new_v2.csv": df_new_stock,
    "complete_timber_A_v2.csv":   df_combined_A,
    "complete_timber_B_v2.csv":   df_combined_B,
}
for filename, df in files.items():
    df.to_csv(config.TIMBER_STOCK_PATH / filename, index=False, sep=';')
    print(f"Saved '{filename}'  ({len(df)} elements  |  "
          f"NS={( df['State'] == 0).sum()}  RS={(df['State'] == 1).sum()})")

print(f"\nNew stock is identical across A, B, and new — {len(df_new_stock)} NS elements shared.")

## 2. Development subset
Generates a small mixed stock file (`complete_timber_small.csv`) for fast development and testing runs in the optimizer. Adjust `total_elements` and `reclaimed_ratio` as needed.

In [ ]:
import config
from c16_generation_timber import generate_mixed_stock_subset

# Small development subsets — generated independently (not required to share NS pool)
for donor in ["A", "B"]:
    df_small = generate_mixed_stock_subset(
        total_elements    = 20,
        reclaimed_ratio   = 0.3,
        random_state      = 38,
        allow_replacement = True,
    )
    filename = f"complete_timber_small_{donor}.csv"
    df_small.to_csv(config.TIMBER_STOCK_PATH / filename, index=False, sep=';')
    print(f"Saved '{filename}'  ({len(df_small)} elements)")